In [ ]:
import requests

session = requests.Session()

session.post('https://www.space-track.org/ajaxauth/login', data={
    'identity': 'dejuakim@gmail.com',
    'password': 'siawevengersaiffel'
})

url = (
    "https://www.space-track.org/basicspacedata/query"
    "/class/gp"
    "/OBJECT_TYPE/PAYLOAD"
    "/MEAN_MOTION/13.7--15.8"
    "/ECCENTRICITY/<0.25"
    "/INCLINATION/>40"
    "/EPOCH/>now-30"
    "/orderby/OBJECT_NAME"
    "/format/json"
)

response = session.get(url)
tle_list = response.json()

print(f"수신된 위성 수: {len(tle_list)}")
print("\n위성 이름 샘플 (처음 30개):")
for sat in tle_list[:30]:
    print(f"  {sat['OBJECT_NAME']:<30} 고도추정: MEAN_MOTION={sat['MEAN_MOTION']}")

수신된 위성 수: 13593

위성 이름 샘플 (처음 30개):
  3CAT-4                         고도추정: MEAN_MOTION=15.02085848
  3CAT-5/A (TYVAK-0161)          고도추정: MEAN_MOTION=15.45143577
  3CAT-5/B (TYVAK-0162)          고도추정: MEAN_MOTION=15.46437482
  6GSTARLAB                      고도추정: MEAN_MOTION=15.18217110
  A-SEANSAT-PG1                  고도추정: MEAN_MOTION=15.35876320
  AAC-AIS-SAT-1                  고도추정: MEAN_MOTION=14.91834702
  AAC-AIS-SAT2                   고도추정: MEAN_MOTION=14.85305100
  AAC-AIS-SAT3                   고도추정: MEAN_MOTION=14.84555624
  AAC-HSI-SAT1                   고도추정: MEAN_MOTION=15.12409015
  AAC-HSI-SAT2                   고도추정: MEAN_MOTION=15.20163723
  AAC-HSI-SAT3                   고도추정: MEAN_MOTION=15.15616608
  AAC-IO1                        고도추정: MEAN_MOTION=14.91379200
  AAU CUBESAT                    고도추정: MEAN_MOTION=14.23945998
  AAUSAT3                        고도추정: MEAN_MOTION=14.39899529
  AC1-001                        고도추정: MEAN_MOTION=15.18205753
  AC1-002          

In [2]:
# OBJECT_NAME 기반이 아닌
# RCS_SIZE 기준 추가 (위성 크기)
# 지구관측 위성은 보통 MEDIUM 또는 LARGE

url = (
    "https://www.space-track.org/basicspacedata/query"
    "/class/gp"
    "/OBJECT_TYPE/PAYLOAD"
    "/MEAN_MOTION/13.7--15.8"
    "/ECCENTRICITY/<0.25"
    "/INCLINATION/>40"
    "/RCS_SIZE/MEDIUM,LARGE"   # 추가
    "/EPOCH/>now-30"
    "/orderby/OBJECT_NAME"
    "/format/json"
)

response = session.get(url)
tle_list_filtered = response.json()
print(f"RCS 필터 후 위성 수: {len(tle_list_filtered)}")

RCS 필터 후 위성 수: 10378


In [3]:
import requests

response = requests.get(
    "https://celestrak.org/SOCRATES/query.php",
    params={"GROUP": "earth-obs", "FORMAT": "tle"}
)
print(f"상태코드: {response.status_code}")
print(response.text[:500])

상태코드: 404
<!DOCTYPE html>
<html>
<head>
<meta http-equiv="Content-Type" content="text/html; charset=iso-8859-1"/>
<title>404 - File Not Found.</title>
<style type="text/css">
body{margin:0;font-size:.7em;font-family:Verdana, Arial, Helvetica, sans-serif;background:#EEEEEE;}
fieldset{padding:0 15px 10px 15px;}
h1{font-size:2.4em;margin:0;color:#FFF;}
h2{font-size:1.7em;margin:0;color:#CC0000;}
h3{font-size:1.2em;margin:10px 0 0 0;color:#000000;}
#header{width:96%;margin:0 0 0 0;padding:6px 2% 6p


In [4]:
import requests

# CelesTrak 새 URL 형식
urls_to_try = [
    "https://celestrak.org/SOCRATES/query.php?GROUP=earth-obs&FORMAT=tle",
    "https://celestrak.org/pub/TLE/catalog.txt",
    "https://celestrak.org/SOCRATES/",
    "https://celestrak.org/pub/TLE/eo-ops.txt",
    "https://celestrak.org/pub/TLE/resource.txt",
]

for url in urls_to_try:
    r = requests.get(url, timeout=10)
    print(f"{r.status_code} | {url}")

404 | https://celestrak.org/SOCRATES/query.php?GROUP=earth-obs&FORMAT=tle
403 | https://celestrak.org/pub/TLE/catalog.txt
200 | https://celestrak.org/SOCRATES/
403 | https://celestrak.org/pub/TLE/eo-ops.txt
403 | https://celestrak.org/pub/TLE/resource.txt


In [5]:
# CelesTrak GP 데이터 API (새 형식)
urls_to_try = [
    "https://celestrak.org/SOCRATES/query.php?GROUP=earth-obs&FORMAT=json",
    "https://celestrak.org/cgi-bin/TLE.cgi?GROUP=earth-obs&FORMAT=tle",
    "https://celestrak.org/SOCRATES/query.php?CATNR=25544&FORMAT=tle",  # ISS 테스트
    "https://celestrak.org/pub/TLE/active.txt",
    "https://celestrak.org/satcat/records.csv",
]

for url in urls_to_try:
    try:
        r = requests.get(url, timeout=10)
        print(f"{r.status_code} | {url}")
    except Exception as e:
        print(f"ERROR | {url} | {e}")

404 | https://celestrak.org/SOCRATES/query.php?GROUP=earth-obs&FORMAT=json
404 | https://celestrak.org/cgi-bin/TLE.cgi?GROUP=earth-obs&FORMAT=tle
404 | https://celestrak.org/SOCRATES/query.php?CATNR=25544&FORMAT=tle
ERROR | https://celestrak.org/pub/TLE/active.txt | HTTPSConnectionPool(host='celestrak.org', port=443): Max retries exceeded with url: /pub/TLE/active.txt (Caused by ConnectTimeoutError(<HTTPSConnection(host='celestrak.org', port=443) at 0x24276eecad0>, 'Connection to celestrak.org timed out. (connect timeout=10)'))
ERROR | https://celestrak.org/satcat/records.csv | HTTPSConnectionPool(host='celestrak.org', port=443): Max retries exceeded with url: /satcat/records.csv (Caused by ConnectTimeoutError(<HTTPSConnection(host='celestrak.org', port=443) at 0x24276ef8d90>, 'Connection to celestrak.org timed out. (connect timeout=10)'))


In [2]:
import pandas as pd
df = pd.read_excel("UCS-Satellite-Database 5-1-2023.xlsx")
print(df.columns.tolist())
print(f"전체 위성 수: {len(df)}")

['Name of Satellite, Alternate Names', 'Current Official Name of Satellite', 'Country/Org of UN Registry', 'Country of Operator/Owner', 'Operator/Owner', 'Users', 'Purpose', 'Detailed Purpose', 'Class of Orbit', 'Type of Orbit', 'Longitude of GEO (degrees)', 'Perigee (km)', 'Apogee (km)', 'Eccentricity', 'Inclination (degrees)', 'Period (minutes)', 'Launch Mass (kg.)', 'Dry Mass (kg.)', 'Power (watts)', 'Date of Launch', 'Expected Lifetime (yrs.)', 'Contractor', 'Country of Contractor', 'Launch Site', 'Launch Vehicle', 'COSPAR Number', 'NORAD Number', 'Comments', 'Unnamed: 28', 'Source Used for Orbital Data', 'Source', 'Source.1', 'Source.2', 'Source.3', 'Source.4', 'Source.5', 'Source.6', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56', '

In [5]:
import pandas as pd

df = pd.read_excel("UCS-Satellite-Database 5-1-2023.xlsx")

# EO + SAR 위성 필터링
eo_sar_df = df[
    df['Detailed Purpose'].str.contains(
        'Optical|Multispectral|Hyperspectral|Video|Imaging|Radar|SAR',
        na=False
    )
][['Current Official Name of Satellite', 'NORAD Number', 'Detailed Purpose']].dropna()

eo_sar_df['NORAD Number'] = eo_sar_df['NORAD Number'].astype(int)

print(f"EO+SAR 위성 수: {len(eo_sar_df)}")
print(eo_sar_df['Detailed Purpose'].value_counts())

norad_ids = eo_sar_df['NORAD Number'].tolist()
print(f"\nNORAD ID 수: {len(norad_ids)}")

EO+SAR 위성 수: 735
Detailed Purpose
Optical Imaging                                          516
Radar Imaging                                             98
Multispectral Imaging                                     25
Hyperspectral Imaging                                     23
Infrared Imaging                                          14
Radar Imaging (SAR)                                       12
Radar Imaging/Earth Science                                8
Video Imaging                                              7
Imaging                                                    5
Optical Imaging/Automatic Identification System (AIS)      3
Optical Imaging/Meterology                                 3
Radar Surveillance                                         2
Synthetic Aperture Radar (SAR)                             2
Synthetic Aperture Imaging                                 2
Optical Imaging (video)                                    2
Optical Imaging/Meteorology                        

In [ ]:
import requests
import json
import time

# Space-Track 로그인
USERNAME = "dejuakim@gmail.com"
PASSWORD = "siawevengersaiffel"

login_url = "https://www.space-track.org/ajaxauth/login"
session = requests.Session()
session.post(login_url, data={"identity": USERNAME, "password": PASSWORD})

# NORAD ID 배치 처리 (100개씩)
def fetch_tle_batch(norad_batch):
    ids = ",".join(str(n) for n in norad_batch)
    url = (
        "https://www.space-track.org/basicspacedata/query"
        f"/class/gp/NORAD_CAT_ID/{ids}"
        "/orderby/NORAD_CAT_ID"
        "/format/json"
    )
    response = session.get(url)
    return response.json()

# 100개씩 나눠서 수집
batch_size = 100
all_tle = []

for i in range(0, len(norad_ids), batch_size):
    batch = norad_ids[i:i+batch_size]
    result = fetch_tle_batch(batch)
    all_tle.extend(result)
    print(f"수집 완료: {i+len(batch)}/{len(norad_ids)}")
    time.sleep(1)  # 요청 간격

print(f"\n총 TLE 수집: {len(all_tle)}개")

# 저장
import os
os.makedirs("project/01_data/raw", exist_ok=True)
with open("../project/01_data/raw/tle_eo_sar.json", "w") as f:
    json.dump(all_tle, f)

print("저장 완료: ../project/01_data/raw/tle_eo_sar.json")

수집 완료: 100/735
수집 완료: 200/735
수집 완료: 300/735
수집 완료: 400/735
수집 완료: 500/735
수집 완료: 600/735
수집 완료: 700/735
수집 완료: 735/735

총 TLE 수집: 693개
저장 완료: project/01_data/raw/tle_eo_sar.json


In [9]:
# 단일 NORAD ID로 테스트
test_id = norad_ids[0]
print(f"테스트 NORAD ID: {test_id}")

url = (
    "https://www.space-track.org/basicspacedata/query"
    f"/class/gp/NORAD_CAT_ID/{test_id}"
    "/orderby/NORAD_CAT_ID"
    "/format/json"
)

response = session.get(url)
print(f"상태 코드: {response.status_code}")
print(response.text[:300])

테스트 NORAD ID: 44859
상태 코드: 200
[{"CCSDS_OMM_VERS":"3.0","COMMENT":"GENERATED VIA SPACE-TRACK.ORG API","CREATION_DATE":"2026-05-20T02:32:39","ORIGINATOR":"18 SPCS","OBJECT_NAME":"IHOPSAT-TD","OBJECT_ID":"2019-089H","CENTER_NAME":"EARTH","REF_FRAME":"TEME","TIME_SYSTEM":"UTC","MEAN_ELEMENT_THEORY":"SGP4","EPOCH":"2026-05-19T11:37:1


In [11]:
import requests
import json
import time

# 배치 처리 함수 수정
def fetch_tle_batch(norad_batch):
    ids = ",".join(str(n) for n in norad_batch)
    url = (
        "https://www.space-track.org/basicspacedata/query"
        f"/class/gp/NORAD_CAT_ID/{ids}"
        "/orderby/NORAD_CAT_ID"
        "/format/json"
    )
    response = session.get(url)
    
    # 오류 체크
    if response.status_code != 200:
        print(f"오류 발생: {response.status_code}")
        return []
    
    result = response.json()
    
    # "error" 문자열 필터링
    if isinstance(result, list):
        result = [r for r in result if isinstance(r, dict)]
    
    return result

# 100개씩 나눠서 수집
batch_size = 100
all_tle = []

for i in range(0, len(norad_ids), batch_size):
    batch = norad_ids[i:i+batch_size]
    result = fetch_tle_batch(batch)
    all_tle.extend(result)
    print(f"수집 완료: {i+len(batch)}/{len(norad_ids)} | 현재 수집: {len(all_tle)}개")
    time.sleep(2)

print(f"\n총 TLE 수집: {len(all_tle)}개")

# 저장
import os
os.makedirs("../project/01_data/raw", exist_ok=True)
with open("../project/01_data/raw/tle_eo_sar.json", "w") as f:
    json.dump(all_tle, f)

print("저장 완료: ../project/01_data/raw/tle_eo_sar.json")

수집 완료: 100/735 | 현재 수집: 98개
수집 완료: 200/735 | 현재 수집: 198개
수집 완료: 300/735 | 현재 수집: 298개
수집 완료: 400/735 | 현재 수집: 384개
수집 완료: 500/735 | 현재 수집: 475개
수집 완료: 600/735 | 현재 수집: 559개
수집 완료: 700/735 | 현재 수집: 658개
수집 완료: 735/735 | 현재 수집: 693개

총 TLE 수집: 693개
저장 완료: ../project/01_data/raw/tle_eo_sar.json
